In [1]:
# ============================================================
# TASK 1 - INTENT CLASSIFICATION
# Vanilla Unidirectional LSTM - Many-to-One
# ============================================================


# ============================================================
# 1. IMPORTS + DRIVE
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import os
import json
import copy
import numpy as np

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence

from collections import Counter


# ============================================================
# 2. PATHS
# ============================================================

FOLDER_PATH = "/content/drive/MyDrive/Calendar-Assistant NLU"

train_path = os.path.join(
    FOLDER_PATH,
    "train.json"
)

val_path = os.path.join(
    FOLDER_PATH,
    "val.json"
)

test_path = os.path.join(
    FOLDER_PATH,
    "test.json"
)


# ============================================================
# 3. LOAD DATA
# ============================================================

with open(train_path, "r") as f:
    train_data = json.load(f)

with open(val_path, "r") as f:
    val_data = json.load(f)

with open(test_path, "r") as f:
    test_data = json.load(f)


print("Training examples  :", len(train_data))
print("Validation examples:", len(val_data))
print("Test examples      :", len(test_data))

print("\nExample:")
print(train_data[0])


# ============================================================
# 4. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:", device)


# ============================================================
# 5. LOAD FASTTEXT
# ============================================================

!pip install -q gensim

import gensim.downloader as api

print("\nLoading FastText...")

fasttext = api.load(
    "fasttext-wiki-news-subwords-300"
)

print("FastText loaded.")


# ============================================================
# 6. BUILD WORD VOCABULARY
# ============================================================
#
# IMPORTANT:
# Vocabulary is constructed ONLY from training data.
# Validation/test words not seen during training become <UNK>.
# ============================================================

counter = Counter()

for sample in train_data:
    counter.update(sample["tokens"])


word2idx = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word in counter:
    if word not in word2idx:
        word2idx[word] = len(word2idx)


idx2word = {
    idx: word
    for word, idx in word2idx.items()
}


print("\nVocabulary size:", len(word2idx))


# ============================================================
# 7. CREATE EMBEDDING MATRIX
# ============================================================

embedding_dim = 300

embedding_matrix = np.random.normal(
    0,
    0.05,
    size=(len(word2idx), embedding_dim)
).astype(np.float32)


# PAD should be zero
embedding_matrix[word2idx["<PAD>"]] = np.zeros(
    embedding_dim,
    dtype=np.float32
)


# Fill vocabulary words using FastText
for word, idx in word2idx.items():

    if word in ["<PAD>", "<UNK>"]:
        continue

    if word in fasttext:
        embedding_matrix[idx] = fasttext[word]


# <UNK> can remain randomly initialized


embedding = nn.Embedding.from_pretrained(
    torch.tensor(embedding_matrix),
    freeze=False
)


print(
    "Embedding shape:",
    embedding.weight.shape
)


# ============================================================
# 8. BUILD INTENT VOCABULARY
# ============================================================

intent2idx = {
    "CREATE_EVENT": 0,
    "SET_REMINDER": 1,
    "QUERY_FREE_TIME": 2,
    "CANCEL": 3
}

idx2intent = {
    idx: intent
    for intent, idx in intent2idx.items()
}


num_classes = len(intent2idx)

print("\nIntent mapping:")
print(intent2idx)


# ============================================================
# 9. CONVERT DATA INTO X AND Y
# ============================================================

def prepare_task1(data):

    X = []
    Y = []

    for sample in data:

        sentence = [
            word2idx.get(
                word,
                word2idx["<UNK>"]
            )
            for word in sample["tokens"]
        ]

        intent = intent2idx[
            sample["intent"]
        ]

        X.append(
            torch.tensor(
                sentence,
                dtype=torch.long
            )
        )

        Y.append(intent)

    return X, torch.tensor(
        Y,
        dtype=torch.long
    )


X_train, Y_train = prepare_task1(train_data)
X_val, Y_val = prepare_task1(val_data)
X_test, Y_test = prepare_task1(test_data)


print("\nPrepared data:")
print("Train:", len(X_train))
print("Val  :", len(X_val))
print("Test :", len(X_test))


# ============================================================
# 10. DATASET
# ============================================================

class Task1Dataset(Dataset):

    def __init__(self, X, Y):
        self.X = X
        self.Y = Y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


train_dataset = Task1Dataset(
    X_train,
    Y_train
)

val_dataset = Task1Dataset(
    X_val,
    Y_val
)

test_dataset = Task1Dataset(
    X_test,
    Y_test
)


# ============================================================
# 11. COLLATE FUNCTION
# ============================================================

def collate_fn(batch):

    X_batch, Y_batch = zip(*batch)

    lengths = torch.tensor(
        [
            len(x)
            for x in X_batch
        ],
        dtype=torch.long
    )

    max_length = max(
        lengths
    )

    padded_X = torch.full(
        (
            len(X_batch),
            max_length
        ),
        word2idx["<PAD>"],
        dtype=torch.long
    )

    for i, x in enumerate(X_batch):

        padded_X[
            i,
            :len(x)
        ] = x

    Y_batch = torch.tensor(
        Y_batch,
        dtype=torch.long
    )

    return (
        padded_X,
        lengths,
        Y_batch
    )


# ============================================================
# 12. DATALOADERS
# ============================================================

batch_size = 32

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_fn
)

val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_fn
)


# ============================================================
# 13. TASK-1 MODEL
# ============================================================
#
# Vanilla unidirectional LSTM
#
# Input:
# [batch, sequence_length, 300]
#
# LSTM:
# [batch, sequence_length, 256]
#
# Final valid hidden state:
# [batch, 256]
#
# FC:
# [batch, 4]
#
# This is MANY-TO-ONE.
# ============================================================

class Task1LSTM(nn.Module):

    def __init__(
        self,
        embedding_layer,
        embedding_dim=300,
        hidden_dim=256,
        num_classes=4
    ):

        super().__init__()

        self.embedding = embedding_layer

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True,
            bidirectional=False
        )

        self.fc = nn.Linear(
            hidden_dim,
            num_classes
        )


    def forward(
        self,
        x,
        lengths
    ):

        # ----------------------------------------
        # Word IDs -> embeddings
        # ----------------------------------------

        x = self.embedding(x)


        # ----------------------------------------
        # Remove PAD influence from LSTM
        # ----------------------------------------

        packed = pack_padded_sequence(
            x,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )


        # ----------------------------------------
        # Vanilla unidirectional LSTM
        # ----------------------------------------

        _, (hidden, cell) = self.lstm(
            packed
        )


        # ----------------------------------------
        # Final hidden state
        #
        # hidden:
        # [1, batch, 256]
        #
        # -> [batch, 256]
        # ----------------------------------------

        hidden = hidden[-1]


        # ----------------------------------------
        # Classification
        # ----------------------------------------

        output = self.fc(
            hidden
        )


        return output


# ============================================================
# 14. CREATE MODEL
# ============================================================

hidden_dim = 256

model1 = Task1LSTM(
    embedding_layer=embedding,
    embedding_dim=embedding_dim,
    hidden_dim=hidden_dim,
    num_classes=num_classes
)

model1 = model1.to(device)

print("\nTask-1 Model:")
print(model1)


# ============================================================
# 15. LOSS + OPTIMIZER
# ============================================================

criterion1 = nn.CrossEntropyLoss()

optimizer1 = torch.optim.Adam(
    model1.parameters(),
    lr=0.001
)


# ============================================================
# 16. TRAINING CONFIGURATION
# ============================================================

num_epochs = 100

# Stop if validation loss does not improve
# for 10 consecutive epochs.
patience = 10

best_val_loss = float("inf")
epochs_without_improvement = 0

best_epoch = 0

best_model_state = None

train_losses_1 = []
val_losses_1 = []


# ============================================================
# 17. TRAINING
# ============================================================

print("\n")
print("=" * 60)
print("TASK-1 TRAINING")
print("=" * 60)


for epoch in range(num_epochs):


    # ========================================================
    # TRAIN
    # ========================================================

    model1.train()

    total_train_loss = 0


    for X_batch, lengths, Y_batch in train_dataloader:

        X_batch = X_batch.to(device)
        lengths = lengths.to(device)
        Y_batch = Y_batch.to(device)


        # Forward
        output = model1(
            X_batch,
            lengths
        )


        # Loss
        loss = criterion1(
            output,
            Y_batch
        )


        # Backpropagation
        optimizer1.zero_grad()

        loss.backward()

        optimizer1.step()


        total_train_loss += loss.item()


    train_loss = (
        total_train_loss /
        len(train_dataloader)
    )

    train_losses_1.append(
        train_loss
    )


    # ========================================================
    # VALIDATION
    # ========================================================

    model1.eval()

    total_val_loss = 0


    with torch.no_grad():

        for X_batch, lengths, Y_batch in val_dataloader:

            X_batch = X_batch.to(device)
            lengths = lengths.to(device)
            Y_batch = Y_batch.to(device)


            output = model1(
                X_batch,
                lengths
            )


            loss = criterion1(
                output,
                Y_batch
            )


            total_val_loss += loss.item()


    val_loss = (
        total_val_loss /
        len(val_dataloader)
    )

    val_losses_1.append(
        val_loss
    )


    # ========================================================
    # PRINT TRAIN + VALIDATION LOSS
    # ========================================================

    print(
        f"Epoch {epoch + 1:3d}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f}"
    )


    # ========================================================
    # BEST MODEL
    # ========================================================

    if val_loss < best_val_loss:

        best_val_loss = val_loss

        best_epoch = epoch + 1

        epochs_without_improvement = 0


        # Save complete model state
        best_model_state = {
            key: value.detach().cpu().clone()
            for key, value
            in model1.state_dict().items()
        }


        print(
            f"   → New best model!"
            f" Val Loss: {best_val_loss:.4f}"
        )


    else:

        epochs_without_improvement += 1

        print(
            f"   → No improvement "
            f"({epochs_without_improvement}/{patience})"
        )


    # ========================================================
    # EARLY STOPPING
    # ========================================================

    if epochs_without_improvement >= patience:

        print(
            "\nEarly stopping triggered."
        )

        print(
            f"Best epoch: {best_epoch}"
        )

        print(
            f"Best validation loss: "
            f"{best_val_loss:.4f}"
        )

        break


# ============================================================
# 18. RESTORE BEST MODEL
# ============================================================

model1.load_state_dict(
    best_model_state
)

model1 = model1.to(device)

model1.eval()


print("\n")
print("=" * 60)
print("BEST TASK-1 MODEL RESTORED")
print("=" * 60)

print(
    "Best Epoch:",
    best_epoch
)

print(
    "Best Validation Loss:",
    best_val_loss
)


# ============================================================
# 19. TASK-1 EVALUATION
# ============================================================
#
# Metrics:
#
# 1. Accuracy for each intent class
# 2. Overall accuracy
#
# Class accuracy:
#
# correct predictions belonging to class
# ----------------------------------------
# total actual samples of that class
#
# Overall:
#
# total correct predictions
# -------------------------
# total test samples
# ============================================================

correct_per_class = {
    intent: 0
    for intent in intent2idx
}

total_per_class = {
    intent: 0
    for intent in intent2idx
}

overall_correct = 0
overall_total = 0


with torch.no_grad():

    for X_batch, lengths, Y_batch in test_dataloader:

        X_batch = X_batch.to(device)
        lengths = lengths.to(device)
        Y_batch = Y_batch.to(device)


        output = model1(
            X_batch,
            lengths
        )


        predictions = output.argmax(
            dim=1
        )


        # ----------------------------------------
        # Overall
        # ----------------------------------------

        overall_correct += (
            predictions == Y_batch
        ).sum().item()

        overall_total += (
            Y_batch.size(0)
        )


        # ----------------------------------------
        # Per-class
        # ----------------------------------------

        for class_idx, intent in idx2intent.items():

            mask = (
                Y_batch == class_idx
            )

            total_per_class[intent] += (
                mask.sum().item()
            )

            correct_per_class[intent] += (
                (
                    predictions[mask]
                    == class_idx
                )
                .sum()
                .item()
            )


# ============================================================
# 20. PRINT EVALUATION
# ============================================================

overall_accuracy = (
    overall_correct /
    overall_total
) * 100


print("\n")
print("=" * 60)
print("TASK-1 EVALUATION")
print("=" * 60)

print(
    f"Overall Accuracy: "
    f"{overall_accuracy:.2f}%"
)

print("-" * 60)

print("Accuracy by Intent:")
print("-" * 60)


for intent in intent2idx:

    correct = correct_per_class[
        intent
    ]

    total = total_per_class[
        intent
    ]

    accuracy = (
        correct / total * 100
        if total > 0
        else 0
    )

    print(
        f"{intent:<18} | "
        f"Correct: {correct:4d} / {total:4d} "
        f"| Accuracy: {accuracy:.2f}%"
    )


print("=" * 60)


# ============================================================
# 21. SAVE COMPLETE TASK-1 CHECKPOINT
# ============================================================

SAVE_DIR = (
    "/content/drive/MyDrive/"
    "Calendar-Assistant NLU"
)

os.makedirs(
    SAVE_DIR,
    exist_ok=True
)


task1_checkpoint = {

    # ========================================================
    # MODEL
    # ========================================================

    "model_state_dict":
        model1.state_dict(),


    # ========================================================
    # EMBEDDING
    # ========================================================

    "embedding_state_dict":
        model1.embedding.state_dict(),


    # ========================================================
    # WORD VOCABULARY
    # ========================================================

    "word2idx":
        word2idx,

    "idx2word":
        idx2word,


    # ========================================================
    # INTENT VOCABULARY
    # ========================================================

    "intent2idx":
        intent2idx,

    "idx2intent":
        idx2intent,


    # ========================================================
    # MODEL CONFIGURATION
    # ========================================================

    "embedding_dim":
        embedding_dim,

    "hidden_dim":
        hidden_dim,

    "num_classes":
        num_classes,

    "batch_size":
        batch_size,


    # ========================================================
    # TRAINING INFORMATION
    # ========================================================

    "best_epoch":
        best_epoch,

    "best_val_loss":
        best_val_loss,


    # ========================================================
    # TEST RESULTS
    # ========================================================

    "overall_accuracy":
        overall_accuracy,

    "correct_per_class":
        correct_per_class,

    "total_per_class":
        total_per_class
}


task1_path = os.path.join(
    SAVE_DIR,
    "task1_final.pt"
)


torch.save(
    task1_checkpoint,
    task1_path
)


print("\n")
print("=" * 60)
print("TASK-1 MODEL SAVED")
print("=" * 60)

print(
    "Location:"
)

print(
    task1_path
)


# ============================================================
# 22. VERIFY SAVED FILE
# ============================================================

loaded_checkpoint = torch.load(
    task1_path,
    map_location="cpu"
)


print("\nCheckpoint verified.")

print("\nSaved contents:")

for key in loaded_checkpoint.keys():
    print(" -", key)

print("\nFile exists:",
      os.path.exists(task1_path))

print("\nFinal location:")
print(task1_path)

Mounted at /content/drive
Training examples  : 3381
Validation examples: 724
Test examples      : 725

Example:
{'raw_text': 'set a reminder to call emma at midnight', 'tokens': ['set', 'a', 'reminder', 'to', 'call', 'emma', 'at', 'midnight'], 'tags': ['O', 'O', 'O', 'O', 'O', 'B-PERSON', 'O', 'B-TIME'], 'intent': 'SET_REMINDER', 'date_iso': None, 'time_hm': '00:00', 'id': 'ex_002698', 'target_string': 'SET_REMINDER|O O O O O B-PERSON O B-TIME|00:00'}

Device: cpu
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 49.7 MB/s eta 0:00:00

Loading FastText...
[==================================================] 100.0% 958.5/958.4MB downloaded
FastText loaded.

Vocabulary size: 649
Embedding shape: torch.Size([649, 300])

Intent mapping:
{'CREATE_EVENT': 0, 'SET_REMINDER': 1, 'QUERY_FREE_TIME': 2, 'CANCEL': 3}

Prepared data:
Train: 3381
Val  : 724
Test : 725

Task-1 Model:
Task1LSTM(
  (embedding): Embedding(649, 300)
  (lstm): LSTM(300, 256, batch_first=True)
  (fc): Linear(in_feat